In [1]:
!pip install -q -U google-genai pillow tqdm pandas pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.5/263.5 kB 8.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.59.1 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import os, json, time, threading, shutil, logging
import numpy as np, cv2, pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from pydantic import BaseModel
from typing import Literal
from google import genai
from google.genai import types

logging.getLogger("google_genai.models").setLevel(logging.ERROR)

# ---------------- Config ----------------
BASE_DIR   = '/content/drive/MyDrive/PRANET-V2-POLYPGEN2021-RESULTS/negativeOnly'
OUT_DIR    = '/content/drive/MyDrive/polyp_audit'          # results go here
CKPT_PATH  = f'{OUT_DIR}/checkpoint.json'
CSV_PATH   = f'{OUT_DIR}/results.csv'
REVIEW_DIR = f'{OUT_DIR}/flagged_for_review'               # copies of flagged images
MODEL      = 'gemini-2.5-flash'
WORKERS    = 8
MAX_RPM    = 60      # use 10 on the free tier
PANEL_SIDE = 768     # max side of each cropped panel sent to Gemini
COPY_FLAGGED = True

assert os.path.isdir(BASE_DIR), "Drive not mounted or BASE_DIR is wrong."
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- API key ----------------
try:
    from google.colab import userdata
    API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    from getpass import getpass
    API_KEY = getpass("Gemini API key: ")
client = genai.Client(api_key=API_KEY)

# ---------------- Prompt / schema ----------------
class Verdict(BaseModel):
    verdict: Literal["POLYP", "NO_POLYP", "UNCERTAIN"]
    confidence: float          # 0.0 - 1.0
    reason: str                # one short sentence

PROMPT = """You are auditing a colonoscopy dataset. This frame was labeled as POLYP-FREE, but the label may be wrong.
I am giving you two images from the same frame:
IMAGE 1: the original colonoscopy frame.
IMAGE 2: the output of an automatic segmentation model (highlighted/colored region = where it thinks a polyp is). {mask_note}

Rules:
- Judge IMAGE 1 on its own. The model's output is often wrong; use it only as a hint of where to look closely. Do not trust it.
- POLYP = a distinct raised, rounded, or pedunculated mass of tissue (smooth or lobulated, often different in color/texture from surrounding mucosa) that is clearly separate from normal anatomy.
- NO_POLYP = only normal mucosa, folds, vessels, bubbles, fluid/stool, glare/specular highlights, or instruments.
- UNCERTAIN = you cannot tell; a human should look.
Return the verdict, a confidence from 0 to 1, and one short sentence of reasoning."""

CFG = types.GenerateContentConfig(
    temperature=0,
    max_output_tokens=300,
    response_mime_type="application/json",
    response_schema=Verdict,
    thinking_config=types.ThinkingConfig(thinking_budget=0),
)

# ---------------- Helpers ----------------
_lock, _next = threading.Lock(), [0.0]
def throttle():
    with _lock:
        now = time.time()
        wait = max(0, _next[0] - now)
        _next[0] = max(now, _next[0]) + 60.0 / MAX_RPM
    if wait: time.sleep(wait)

def split_panels(path):
    img = Image.open(path).convert('RGB')
    w, h = img.size
    third = w // 3
    orig = img.crop((0, 0, third, h))
    pred = img.crop((2 * third, 0, w, h))
    for p in (orig, pred):
        p.thumbnail((PANEL_SIDE, PANEL_SIDE))
    return orig, pred

def pred_has_mask(pred_img):
    hsv = cv2.cvtColor(np.array(pred_img), cv2.COLOR_RGB2HSV)
    m = cv2.inRange(hsv, np.array([100, 50, 50]), np.array([140, 255, 255]))
    return int(cv2.countNonZero(m)) > 300

def audit(path):
    try:
        orig, pred = split_panels(path)
    except Exception as e:
        return {"verdict": "ERROR", "confidence": 0, "reason": f"read failed: {e}", "pred_mask": None}
    has_mask = pred_has_mask(pred)
    note = ("The model DID highlight a region." if has_mask
            else "The model highlighted nothing (it may have missed something).")
    prompt = PROMPT.format(mask_note=note)
    for attempt in range(4):
        try:
            throttle()
            r = client.models.generate_content(
                model=MODEL, contents=[prompt, orig, pred], config=CFG)
            d = json.loads(r.text)
            d["pred_mask"] = has_mask
            return d
        except Exception as e:
            if attempt == 3:
                return {"verdict": "ERROR", "confidence": 0, "reason": str(e)[:200], "pred_mask": has_mask}
            time.sleep(2 ** (attempt + 1))

# ---------------- Gather files (seq1..seq23) ----------------
files = sorted(os.path.join(r, f) for r, _, fs in os.walk(BASE_DIR)
               for f in fs if f.lower().endswith(('.jpg', '.jpeg', '.png')))
print(f"Total images found: {len(files)}")

done = json.load(open(CKPT_PATH)) if os.path.exists(CKPT_PATH) else {}
# retry earlier errors
done = {k: v for k, v in done.items() if v.get("verdict") != "ERROR"}
todo = [f for f in files if f not in done]
print(f"Already done: {len(done)} | To process: {len(todo)}")

# ---------------- Run ----------------
with ThreadPoolExecutor(WORKERS) as ex:
    futs = {ex.submit(audit, p): p for p in todo}
    for i, fut in enumerate(tqdm(as_completed(futs), total=len(futs), desc="Gemini audit")):
        p = futs[fut]
        done[p] = fut.result()
        if done[p]["verdict"] == "POLYP":
            tqdm.write(f"🚨 POLYP ({done[p]['confidence']:.2f}): {p}")
        if i % 25 == 0:
            json.dump(done, open(CKPT_PATH, 'w'))
json.dump(done, open(CKPT_PATH, 'w'))

# ---------------- Report ----------------
rows = [{"seq_folder": os.path.basename(os.path.dirname(p)), "file": os.path.basename(p),
         "full_path": p, **v} for p, v in done.items()]
df = pd.DataFrame(rows).sort_values(["seq_folder", "file"])
df.to_csv(CSV_PATH, index=False)

flagged = df[df.verdict.isin(["POLYP", "UNCERTAIN"])]
print("\nVerdict counts:\n", df.verdict.value_counts().to_string())

print("\n===== LIKELY HIDDEN POLYPS =====")
for seq, g in df[df.verdict == "POLYP"].groupby("seq_folder"):
    print(f"\n[{seq}]  ({len(g)} images)")
    for _, r in g.iterrows():
        print(f"  {r.full_path}   conf={r.confidence:.2f}  | {r.reason}")

print("\n(UNCERTAIN images are in results.csv - worth a quick manual look.)")
print(f"Full results: {CSV_PATH}")

if COPY_FLAGGED:
    for _, r in flagged.iterrows():
        dst = os.path.join(REVIEW_DIR, r.verdict, r.seq_folder)
        os.makedirs(dst, exist_ok=True)
        shutil.copy2(r.full_path, dst)
    print(f"Flagged images copied to: {REVIEW_DIR}")

AssertionError: Drive not mounted or BASE_DIR is wrong.

In [ ]:
if "RESOURCE_EXHAUSTED" in str(e) or "429" in str(e):
    if "PerDay" in str(e) or "per day" in str(e).lower():
        return {"verdict": "ERROR", "confidence": 0, "reason": "DAILY_QUOTA", "pred_mask": has_mask}